In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path

In [2]:
BASE_DIR = Path("..").resolve()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUT_DIR = BASE_DIR / "outputs" / "part2"

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
df = pd.read_csv(PROCESSED_DIR / "audusd_strategy_data.csv", parse_dates=["Date"])

print(df.shape)
df.head()

(179, 15)


,Date,spot,fx_return,rate_au,rate_us,cpi_au,cpi_us,infl_au,infl_us,infl_diff,rate_diff,signal,carry_component,fx_component,strategy_return
0,2009-02-28,0.642219,0.005827,3.35,0.16,64.206667,212.193,0.000363,0.004961,-0.004597,3.19,1,0.002658,0.005827,0.008485
1,2009-03-31,0.691802,0.074370,3.25,0.17,64.230000,212.709,0.000363,0.002429,-0.002065,3.08,1,0.002567,0.074370,0.076937
2,2009-04-30,0.726691,0.049202,3.06,0.04,64.336667,213.240,0.001659,0.002493,-0.000834,3.02,1,0.002517,0.049202,0.051719
3,2009-05-31,0.801025,0.097391,3.00,0.14,64.443333,213.856,0.001657,0.002885,-0.001228,2.86,1,0.002383,0.097391,0.099774
4,2009-06-30,0.806192,0.006429,3.00,0.17,64.550000,215.693,0.001654,0.008553,-0.006899,2.83,1,0.002358,0.006429,0.008787


In [4]:
df["actual_fx_change"] = df["fx_return"]
df["uip_implied_change"] = (df["rate_au"] - df["rate_us"]) / 100 / 12
df["ppp_implied_change"] = df["infl_diff"]

part2 = df[[
    "Date",
    "actual_fx_change",
    "uip_implied_change",
    "ppp_implied_change"
]].copy()

part2.head()

,Date,actual_fx_change,uip_implied_change,ppp_implied_change
0,2009-02-28,0.005827,0.002658,-0.004597
1,2009-03-31,0.074370,0.002567,-0.002065
2,2009-04-30,0.049202,0.002517,-0.000834
3,2009-05-31,0.097391,0.002383,-0.001228
4,2009-06-30,0.006429,0.002358,-0.006899


In [5]:
part2_is = part2[
    (part2["Date"] >= "2009-02-01") &
    (part2["Date"] <= "2019-12-31")
].dropna().copy()

print(part2_is.shape)
part2_is.head()
part2_is.tail()

(131, 4)


,Date,actual_fx_change,uip_implied_change,ppp_implied_change
126,2019-08-31,-0.020483,-0.000917,0.001927
127,2019-09-30,0.003712,-0.000758,0.001090
128,2019-10-31,0.020063,-0.000692,0.000001
129,2019-11-30,-0.018861,-0.000725,0.002816
130,2019-12-31,0.033757,-0.000608,0.003185


In [6]:
X_uip = sm.add_constant(part2_is["uip_implied_change"])
y = part2_is["actual_fx_change"]

uip_model = sm.OLS(y, X_uip).fit()
print(uip_model.summary())

                            OLS Regression Results                            
Dep. Variable:       actual_fx_change   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     1.066
Date:                Wed, 06 May 2026   Prob (F-statistic):              0.304
Time:                        23:45:39   Log-Likelihood:                 257.73
No. Observations:                 131   AIC:                            -511.5
Df Residuals:                     129   BIC:                            -505.7
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -0.0031      0

In [7]:
uip_results = pd.DataFrame({
    "term": uip_model.params.index,
    "coef": uip_model.params.values,
    "p_value": uip_model.pvalues.values,
    "ci_low": uip_model.conf_int()[0].values,
    "ci_high": uip_model.conf_int()[1].values
})

uip_r2 = uip_model.rsquared

print("UIP R-squared:", uip_r2)
uip_results

UIP R-squared: 0.008193458287156674


,term,coef,p_value,ci_low,ci_high
0,const,-0.003082,0.515333,-0.012429,0.006265
1,uip_implied_change,2.119239,0.303853,-1.942447,6.180925


In [8]:
X_ppp = sm.add_constant(part2_is["ppp_implied_change"])
y = part2_is["actual_fx_change"]

ppp_model = sm.OLS(y, X_ppp).fit()
print(ppp_model.summary())

                            OLS Regression Results                            
Dep. Variable:       actual_fx_change   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                 -0.004
Method:                 Least Squares   F-statistic:                    0.4710
Date:                Wed, 06 May 2026   Prob (F-statistic):              0.494
Time:                        23:45:55   Log-Likelihood:                 257.43
No. Observations:                 131   AIC:                            -510.9
Df Residuals:                     129   BIC:                            -505.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  0.0009      0

In [9]:
ppp_results = pd.DataFrame({
    "term": ppp_model.params.index,
    "coef": ppp_model.params.values,
    "p_value": ppp_model.pvalues.values,
    "ci_low": ppp_model.conf_int()[0].values,
    "ci_high": ppp_model.conf_int()[1].values
})

ppp_r2 = ppp_model.rsquared

print("PPP R-squared:", ppp_r2)
ppp_results

PPP R-squared: 0.0036377873765696123


,term,coef,p_value,ci_low,ci_high
0,const,0.000876,0.770470,-0.005052,0.006804
1,ppp_implied_change,-0.693403,0.493765,-2.692448,1.305641


In [11]:
summary = pd.DataFrame({
    "model": ["UIP", "PPP"],
    "alpha": [uip_model.params["const"], ppp_model.params["const"]],
    "alpha_pvalue": [uip_model.pvalues["const"], ppp_model.pvalues["const"]],
    "beta": [
        uip_model.params["uip_implied_change"],
        ppp_model.params["ppp_implied_change"]
    ],
    "beta_pvalue": [
        uip_model.pvalues["uip_implied_change"],
        ppp_model.pvalues["ppp_implied_change"]
    ],
    "r_squared": [uip_model.rsquared, ppp_model.rsquared]
})

summary

,model,alpha,alpha_pvalue,beta,beta_pvalue,r_squared
0,UIP,-0.003082,0.515333,2.119239,0.303853,0.008193
1,PPP,0.000876,0.770470,-0.693403,0.493765,0.003638


In [12]:
theory_table = pd.DataFrame({
    "model": ["UIP", "PPP"],
    "theoretical_alpha": [0, 0],
    "theoretical_beta": [1, 1],
    "null_hypothesis": [
        "H0: alpha = 0 and beta = 1",
        "H0: alpha = 0 and beta = 1"
    ],
    "alternative_hypothesis": [
        "H1: alpha != 0 and/or beta != 1",
        "H1: alpha != 0 and/or beta != 1"
    ]
})

theory_table

,model,theoretical_alpha,theoretical_beta,null_hypothesis,alternative_hypothesis
0,UIP,0,1,H0: alpha = 0 and beta = 1,H1: alpha != 0 and/or beta != 1
1,PPP,0,1,H0: alpha = 0 and beta = 1,H1: alpha != 0 and/or beta != 1


In [13]:
beta_ci = pd.DataFrame({
    "model": ["UIP", "PPP"],
    "beta_hat": [
        uip_model.params["uip_implied_change"],
        ppp_model.params["ppp_implied_change"]
    ],
    "beta_ci_low": [
        uip_model.conf_int().loc["uip_implied_change", 0],
        ppp_model.conf_int().loc["ppp_implied_change", 0]
    ],
    "beta_ci_high": [
        uip_model.conf_int().loc["uip_implied_change", 1],
        ppp_model.conf_int().loc["ppp_implied_change", 1]
    ]
})

beta_ci

,model,beta_hat,beta_ci_low,beta_ci_high
0,UIP,2.119239,-1.942447,6.180925
1,PPP,-0.693403,-2.692448,1.305641


In [14]:
uip_results.to_csv(OUT_DIR / "uip_regression_results.csv", index=False)
ppp_results.to_csv(OUT_DIR / "ppp_regression_results.csv", index=False)
summary.to_csv(OUT_DIR / "part2_regression_summary.csv", index=False)
theory_table.to_csv(OUT_DIR / "part2_theory_table.csv", index=False)
beta_ci.to_csv(OUT_DIR / "part2_beta_confidence_intervals.csv", index=False)